# 📄 PDF Full-Text Extraction

Extracts all text from a scanned PDF and saves it as a single `.txt` file.

No AI, no article splitting — just the raw text of every page, in order.

**Install dependencies first (run once in your terminal):**
```bash
pip install pymupdf
```

---

## Cell 1 — Configuration

Set the two variables below before running:

| Variable | What it does |
|---|---|
| `PDF_PATH` | The filename (or full path) of the PDF you want to extract text from |
| `OUTPUT_FILE` | The name of the `.txt` file that will be created. If left blank, it defaults to the same name as the PDF |

> **Note:** If the PDF is a pure scan with no embedded text layer (i.e. it's just images), PyMuPDF will return empty pages. See the note at the bottom of Cell 3 on how to detect this.

In [1]:
# ── CELL 1: Configuration ─────────────────────────────────────────────────

# Path to your PDF
PDF_PATH = "/workspaces/LRLL_MalawiWorkshopPDFtoMD/TK1E2013_01_03_Page05.pdf"   # <-- change this

# Output text file. Leave as "" to auto-name from the PDF filename.
OUTPUT_FILE = ""   # e.g. "extracted_text.txt", or leave blank

## Cell 2 — Extract and Save

This cell does all the work:

1. **Opens the PDF** using PyMuPDF (`fitz`) — the same library used in the Claude pipeline
2. **Loops through every page** and calls `page.get_text()`, which returns all the text PyMuPDF can find on that page
3. **Adds a page header** before each page's text (e.g. `=== Page 1 ===`) so you can find your place in the output file
4. **Writes everything to a single `.txt` file** in UTF-8 encoding, which correctly handles Chichewa characters

The output is one long text file with all pages concatenated in order.

> **What is `get_text()`?** PyMuPDF reads the text layer that is embedded in the PDF file — the same layer a PDF viewer uses when you select and copy text. If the PDF was created digitally (e.g. exported from InDesign), this works perfectly. If it's a scanned image with no text layer, `get_text()` returns an empty string for that page — see the warning printed at the end of the run.

In [2]:
# ── CELL 2: Extract and Save ───────────────────────────────────────────────

import os
import fitz  # PyMuPDF

# Auto-name the output file if not specified
if not OUTPUT_FILE:
    base = os.path.splitext(os.path.basename(PDF_PATH))[0]
    OUTPUT_FILE = f"{base}.txt"

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found: {PDF_PATH} — check PDF_PATH in Cell 1")

doc = fitz.open(PDF_PATH)
print(f"Opened: {PDF_PATH}  ({len(doc)} pages)")
print(f"Output file: {OUTPUT_FILE}\n")

all_text = []
empty_pages = []

for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text()

    page_text = f"=== Page {page_num + 1} ===\n\n{text.strip()}"
    all_text.append(page_text)

    word_count = len(text.split())
    print(f"Page {page_num + 1}/{len(doc)} — {word_count} words")

    if word_count == 0:
        empty_pages.append(page_num + 1)

doc.close()

# Write to file
full_text = "\n\n".join(all_text)
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(full_text)

total_words = len(full_text.split())
print()
print("─" * 50)

if empty_pages:
    print(f"⚠️  {len(empty_pages)} page(s) returned no text: {empty_pages}")
    print("   These pages may be scanned images with no embedded text layer.")
    print("   If this is the case, you will need OCR — see note below.")

print(f"✅  Done — {total_words:,} words saved to '{OUTPUT_FILE}'")

Opened: /workspaces/LRLL_MalawiWorkshopPDFtoMD/TK1E2013_01_03_Page05.pdf  (1 pages)
Output file: TK1E2013_01_03_Page05.txt

Page 1/1 — 543 words

──────────────────────────────────────────────────
✅  Done — 547 words saved to 'TK1E2013_01_03_Page05.txt'


---

## If your PDF returns empty pages (scanned images)

Some PDFs — especially older newspaper scans — contain only images with no embedded text. In that case, `get_text()` returns nothing and you need **OCR** (Optical Character Recognition) to extract the text.

Install the OCR tools:
```bash
pip install pytesseract pillow pymupdf
sudo apt install tesseract-ocr   # Linux / Codespaces
# or: brew install tesseract     # macOS
```

Then replace `page.get_text()` in Cell 2 with:

```python
import pytesseract
from PIL import Image
import io

# Render the page as an image, then OCR it
mat = fitz.Matrix(2, 2)  # 2x scale = ~144 DPI, good for OCR
pix = page.get_pixmap(matrix=mat)
img = Image.open(io.BytesIO(pix.tobytes("png")))
text = pytesseract.image_to_string(img, lang="eng")  # change lang if needed
```

> For Chichewa text, Tesseract's English model (`lang="eng"`) works reasonably well since Chichewa uses the Latin alphabet. A custom trained model would give better results for production use.